# Clinical Guidelines Q&A — RAG Pipeline

This project builds a question-answering system on top of real clinical guidelines.

You can ask it plain English questions like "what is the first line treatment for type 2 diabetes?" and it finds the relevant section in the document and gives you a sourced answer.

The system uses a technique called RAG (Retrieval-Augmented Generation). Instead of feeding the entire 112-page document to an AI model every time, it first finds the most relevant paragraphs, then passes only those to the model to generate an answer. This makes it faster, cheaper, and more accurate.

**Document used:** IDF Global Clinical Practice Recommendations for Managing Type 2 Diabetes — 2025  
**Model:** Claude Sonnet (Anthropic)  
**Vector store:** ChromaDB (local)  
**Embeddings:** all-MiniLM-L6-v2 (runs locally, no API needed)


## Step 1 — Load the PDF

We read the PDF and extract all the text from it. The IDF guidelines document has 112 pages and around 327,927 characters of text.


In [3]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
reader = PdfReader("../data/diabetes_guidelines.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

print(f"Total pages: {len(reader.pages)}")
print(f"Total characters: {len(text)}")
print(text[:500])

Total pages: 112
Total characters: 327907
IDF Global Clinical Practice 
Recommendations for 
Managing Type 2 Diabetes
2025
IDF Global Clinical Practice 
Recommendations for  
Managing Type 2 Diabetes 
2025In Memoriam 
We are deeply saddened by the passing of Professor Akhtar Hussain (IDF President 2022-2024),  
who served as Chair of this initiative and led its development. His leadership, dedication,  
and expertise significantly shaped this publication. We respectfully dedicate this work to his memory.
Correspondence
Prof. Stephen Col


## Step 2 — Split the Text into Chunks

A language model can only read a limited amount of text at once. So instead of passing all 327,000 characters every time someone asks a question, we split the document into small overlapping pieces of around 500 characters each.

The overlap between chunks (50 characters) makes sure that sentences at the boundary between two chunks don't get cut off and lose their meaning.

This gives us 748 chunks that we can search through efficiently.


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_text(text)

print(f"Total chunks: {len(chunks)}")
print(f"Average chunk size: {sum(len(c) for c in chunks) // len(chunks)} characters")
print("\n--- Sample chunk ---")
print(chunks[10])

Total chunks: 748
Average chunk size: 473 characters

--- Sample chunk ---
Recommendations for Managing Type 2 Diabetes. 
These recommendations reflect our ongoing 
commitment to improving the lives of people 
living with diabetes worldwide by equipping 
healthcare providers with the tools to help them 
deliver high-quality, evidence-based care.  
Diabetes remains a serious and fast-growing 
public health challenge. The IDF Diabetes Atlas 
estimated that 589 million adults aged 20–79 
years were living with diabetes in 2024 and that


## Step 3 — Convert Chunks to Vectors and Store Them

Each chunk gets converted into a list of 384 numbers that represents its meaning. Two chunks that are about similar topics will have similar numbers. This is called an embedding.

We use a free local model (`all-MiniLM-L6-v2`) to generate the embeddings — no API calls needed for this step.

All 748 embeddings get stored in ChromaDB, a local vector database saved to the `data/chroma_db` folder. This only needs to run once — after that we just load the existing database.


In [6]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embedding_model,
    persist_directory="../data/chroma_db"
)

print(f"Vector store created with {vectorstore._collection.count()} chunks")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_30152\92791129.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\ASUS\AppData\Local\Temp\ipykernel_30152\92791129.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\venvs\daily\Lib\site-packages\torch\cuda\__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11060). Please update yo

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created with 2244 chunks


## Step 4 — Build the Retriever

The retriever is the search engine. When a question comes in, it converts the question into a vector using the same embedding model, then finds the 5 chunks in the database whose vectors are closest to the question vector.

Closest vector = most similar meaning. This is how it finds relevant content even when the exact words don't match.


In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

query = "What HbA1c level is recommended for diagnosis of diabetes?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} chunks\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print()

Retrieved 5 chunks

--- Chunk 1 ---
termed “prediabetes” was adopted by ADA in 
2010, defined as an HbA1c of 5.7%–6.4% (39–47 
mmol/mol).19 To date, WHO has not made a 
recommendation on HbA1c for IH. Other groups IDF Global Clinical Practice Recommendations for Managing Type 2 Diabetes – 2025
24
SCREENING AND PREVENTION
such as the UK National Institute for Health 
and Care Excellence recommends an HbA1c of 
6.0%–6.4% (42–47 mmol/mol).20  
Prevalence of intermediate hyperglycaemia

--- Chunk 2 ---
termed “prediabetes” was adopted by ADA in 
2010, defined as an HbA1c of 5.7%–6.4% (39–47 
mmol/mol).19 To date, WHO has not made a 
recommendation on HbA1c for IH. Other groups IDF Global Clinical Practice Recommendations for Managing Type 2 Diabetes – 2025
24
SCREENING AND PREVENTION
such as the UK National Institute for Health 
and Care Excellence recommends an HbA1c of 
6.0%–6.4% (42–47 mmol/mol).20  
Prevalence of intermediate hyperglycaemia

--- Chunk 3 ---
termed “prediabetes” was ado

## Step 5 — Generate Answers with Claude

Now we connect everything together. For each question:

1. The retriever finds the 5 most relevant chunks from the guidelines
2. Those chunks get passed to Claude as context
3. Claude reads the context and writes a clean, sourced answer

The prompt tells Claude to answer only from the provided context and to always cite where the information came from. This prevents the model from making things up — it can only use what's in the document.

**Note:** You need an Anthropic API key to run this step. Set it in your `.env` file as `ANTHROPIC_API_KEY=your-key-here`.


In [8]:
from dotenv import load_dotenv
import os
import anthropic

load_dotenv(dotenv_path="a:/RAG/.env")

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
def ask_guidelines(question):
    # Step 1 - retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Step 2 - build the prompt
    prompt = f"""You are a clinical assistant. Answer the question below using only the context provided.
If the answer is not in the context, say "I could not find this in the guidelines."
Always end your answer by mentioning which topic or section the information came from.

Context:
{context}

Question:
{question}"""
    
    # Step 3 - call Claude
    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return message.content[0].text


# Test it
answer = ask_guidelines("What HbA1c level is recommended for diagnosis of diabetes?")
print(answer)

Based on the provided context, I could not find a specific HbA1c threshold recommended for the **diagnosis of diabetes** itself.

The context only outlines HbA1c ranges used to define **intermediate hyperglycaemia (prediabetes)**:

- **ADA (2010):** HbA1c of **5.7%–6.4%** (39–47 mmol/mol)
- **NICE (UK):** HbA1c of **6.0%–6.4%** (42–47 mmol/mol)
- **WHO:** Has **not made a recommendation** on HbA1c for intermediate hyperglycaemia

For the diagnostic HbA1c threshold for diabetes (typically ≥6.5%), this specific information is **not detailed in the provided context**, though a WHO reference on the use of HbA1c in diagnosis of diabetes mellitus is cited (WHO/NMH/CHP/CPM/11.1, 2011).

📌 *This information comes from the **Screening and Prevention** section of the **IDF Global Clinical Practice Recommendations for Managing Type 2 Diabetes – 2025**.*


## Step 6 — Test with Real Clinical Questions

We test the pipeline with three questions covering different parts of the guidelines: diagnosis criteria, pharmacological treatment, and lifestyle interventions.


In [9]:
answer = ask_guidelines("What are the first line medications for treating type 2 diabetes?")
print(answer)

## First-Line Medications for Type 2 Diabetes

Based on the context provided, the information specifically addresses treatment recommendations when patients are **already on metformin or combination therapy**, suggesting that **metformin** is an established foundational treatment.

The guidelines indicate that:

- **If only on metformin** → Add an **SGLT2 inhibitor** or any available blood glucose-lowering (BGL) therapy
- **If on combination therapy** → Add an **SGLT2 inhibitor** or any available BGL therapy

> 📝 **Note:** The context provided does not explicitly detail the *initial* first-line medications from the beginning of treatment. It focuses on **add-on therapy** recommendations beyond metformin.

It is worth noting that **SGLT2 inhibitors** are highlighted as a preferred add-on option, with the guidelines noting they are *"increasingly available in several low- and middle-income countries at generally affordable cost."*

⚠️ A complete answer regarding first-line therapy initia

In [10]:
answer = ask_guidelines("What lifestyle interventions are recommended for diabetes prevention?")
print(answer)

## Lifestyle Interventions Recommended for Diabetes Prevention

Based on the provided context, the following lifestyle interventions are recommended for diabetes prevention:

### Individual-Level Interventions:
- **Weight reduction**
- **Increased physical activity**

### Population-Level Approaches:
- **Healthier eating**
- **Regular physical activity**
- **Preventing and reducing modifiable diabetes risk factors**

### Broader Population-Based Measures:
- Improving **general education**
- Reducing **poverty**
- Ensuring **food security**
- Raising **health awareness**

The context also notes that:
> *"A decision to implement a high-risk individual diabetes prevention programme should be guided by local resources, health system capacity, and cost. Local design and adaptation are important."*

It is further supported by **randomised controlled studies** that have demonstrated the effective prevention of Type 2 Diabetes Mellitus (T2DM) through these interventions, which can be incorpora

## What This Demonstrates

- How to build a RAG pipeline from scratch using real clinical documents
- How vector search finds relevant content based on meaning, not just keywords
- How to use a language model as a reasoning layer on top of retrieved documents
- How to keep the model grounded to a source document to prevent hallucination

This same pattern is used in production health tech systems that let clinicians query medical literature, hospital policies, or drug databases in plain English.
